# DuckDB GDELT Event Analysis

This notebook demonstrates how to connect to a DuckDB database and query data from the `gedelt_event` table.

## Overview
- Connect to an existing DuckDB database
- Explore the structure of the gedelt_event table
- Perform sample queries and analysis
- Visualize GDELT event data

## 1. Import Required Libraries

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")
print(f"DuckDB version: {duckdb.__version__}")

## 2. Connect to DuckDB Database

DuckDB doesn't have traditional client-server architecture like PostgreSQL. However, you can:

1. **File Connection**: Connect directly to a DuckDB database file
2. **Shared Access**: Multiple processes can read from the same DuckDB file
3. **Read-Only Mode**: Connect to a file that another process is using
4. **MotherDuck**: Connect to cloud-hosted DuckDB (if using MotherDuck)

Choose your connection method below.

In [ ]:
# =============================================================================
# DuckDB Connection Options
# =============================================================================

# Connection configuration
CONNECTION_TYPE = "file"  # Options: "file", "readonly", "motherduck"
DB_FILE_PATH = '../../etl/duckdatabase'  # Path to your DuckDB file
MOTHERDUCK_TOKEN = ""  # Your MotherDuck token (if using MotherDuck)

conn = None

if CONNECTION_TYPE == "file":
    # Standard file connection - can read/write if no other process has exclusive lock
    try:
        conn = duckdb.connect(DB_FILE_PATH)
        print(f"✅ Connected to DuckDB file: {DB_FILE_PATH}")
        
    except Exception as e:
        print(f"❌ File connection failed: {e}")
        print("The database file might be locked by another process.")
        print("Trying read-only connection...")
        
        # Try read-only connection if file is locked
        try:
            conn = duckdb.connect(DB_FILE_PATH, read_only=True)
            print(f"✅ Connected to DuckDB file in READ-ONLY mode: {DB_FILE_PATH}")
            
        except Exception as readonly_error:
            print(f"❌ Read-only connection also failed: {readonly_error}")

elif CONNECTION_TYPE == "readonly":
    # Read-only connection - safe when another process is using the database
    try:
        conn = duckdb.connect(DB_FILE_PATH, read_only=True)
        print(f"✅ Connected to DuckDB file in READ-ONLY mode: {DB_FILE_PATH}")
        print("Note: You won't be able to modify data in read-only mode.")
        
    except Exception as e:
        print(f"❌ Read-only connection failed: {e}")

elif CONNECTION_TYPE == "motherduck":
    # MotherDuck cloud connection
    if not MOTHERDUCK_TOKEN:
        print("❌ MotherDuck token required. Please set MOTHERDUCK_TOKEN.")
    else:
        try:
            # Install motherduck extension if needed
            temp_conn = duckdb.connect()
            temp_conn.execute("INSTALL motherduck")
            temp_conn.execute("LOAD motherduck")
            temp_conn.close()
            
            # Connect to MotherDuck
            conn = duckdb.connect(f"md:{MOTHERDUCK_TOKEN}")
            print("✅ Connected to MotherDuck cloud database")
            
        except Exception as e:
            print(f"❌ MotherDuck connection failed: {e}")

# Test the connection
if conn is not None:
    try:
        # Test basic query
        result = conn.execute("SELECT current_timestamp as connection_time").fetchone()
        print(f"Connection established at: {result[0]}")
        
        # Get DuckDB version
        version_result = conn.execute("SELECT version()").fetchone()
        print(f"DuckDB version: {version_result[0]}")
        
        # Check if connection is read-only
        try:
            conn.execute("CREATE TEMP TABLE test_write (id INTEGER)")
            conn.execute("DROP TABLE test_write")
            print("Connection mode: READ-WRITE")
        except:
            print("Connection mode: READ-ONLY")
            
    except Exception as e:
        print(f"❌ Connection test failed: {e}")
else:
    print("❌ No database connection established")
    print("\nTroubleshooting tips:")
    print("1. Check if the database file path is correct")
    print("2. Ensure the file exists and is accessible")
    print("3. If another process is using the database, try CONNECTION_TYPE = 'readonly'")
    print("4. Check file permissions")

## 3. Explore Database Structure

Let's examine what tables are available and explore the structure of the gedelt_event table.

In [ ]:
# List all tables in the database
tables_query = "SHOW TABLES"
tables_df = conn.execute(tables_query).df()
print("Available tables in the database:")
print(tables_df)

In [ ]:
# Check if gedelt_event table exists and get its structure
try:
    schema_query = "DESCRIBE gdelt_events"
    schema_df = conn.execute(schema_query).df()
    print("Structure of gedelt_event table:")
    print(schema_df)
    
    # Get row count
    count_query = "SELECT COUNT(*) as total_events FROM gdelt_events"
    count_result = conn.execute(count_query).fetchone()
    print(f"\nTotal events in gedelt_event table: {count_result[0]:,}")
    
except Exception as e:
    print(f"Error accessing gedelt_event table: {e}")
    print("The table might not exist or have a different name.")

## 4. Sample Data Exploration

Let's look at a sample of the data to understand its structure.

In [ ]:
# Get a sample of the data
sample_query = """
SELECT * 
FROM gdelt_events
LIMIT 10
"""

try:
    sample_df = conn.execute(sample_query).df()
    print("Sample data from gedelt_event table:")
    print(sample_df.head())
    
    print("\nData types:")
    print(sample_df.dtypes)
    
except Exception as e:
    print(f"Error fetching sample data: {e}")

## 5. Date Range Analysis

Analyze the date range of events in the database.

In [ ]:
# Analyze date range (assuming there's a date column - adjust column name as needed)
date_range_query = """
SELECT 
    MIN(SQLDATE) as earliest_date,
    MAX(SQLDATE) as latest_date,
    COUNT(DISTINCT SQLDATE) as unique_dates,
    COUNT(*) as total_events
FROM gdelt_events
"""

try:
    date_range_df = conn.execute(date_range_query).df()
    print("Date range analysis:")
    print(date_range_df)
    
except Exception as e:
    print(f"Error analyzing date range: {e}")
    print("Note: Adjust the column name if your date column has a different name")

## 6. Event Type Analysis

Analyze different types of events in the GDELT data.

In [ ]:
# Analyze event types (using EventCode if available)
event_types_query = """
SELECT 
    EventCode,
    EventBaseCode,
    COUNT(*) as event_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM gdelt_events
GROUP BY EventCode, EventBaseCode
ORDER BY event_count DESC
LIMIT 20
"""

try:
    event_types_df = conn.execute(event_types_query).df()
    print("Top 20 Event Types:")
    print(event_types_df)
    
    # Create a bar chart
    plt.figure(figsize=(12, 8))
    plt.bar(range(len(event_types_df)), event_types_df['event_count'])
    plt.xlabel('Event Code')
    plt.ylabel('Number of Events')
    plt.title('Distribution of GDELT Event Types')
    plt.xticks(range(len(event_types_df)), event_types_df['EventCode'], rotation=45)
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Error analyzing event types: {e}")

## 7. Geographic Analysis

Analyze events by country and location.

In [ ]:
# Analyze events by country
country_analysis_query = """
SELECT 
    Actor1CountryCode as country,
    COUNT(*) as event_count,
    AVG(GoldsteinScale) as avg_goldstein_scale
FROM gdelt_events
WHERE Actor1CountryCode IS NOT NULL
GROUP BY Actor1CountryCode
ORDER BY event_count DESC
LIMIT 15
"""

try:
    country_df = conn.execute(country_analysis_query).df()
    print("Top 15 Countries by Event Count:")
    print(country_df)
    
    # Create visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Event count by country
    ax1.bar(country_df['country'], country_df['event_count'])
    ax1.set_xlabel('Country Code')
    ax1.set_ylabel('Number of Events')
    ax1.set_title('Events by Country')
    ax1.tick_params(axis='x', rotation=45)
    
    # Average Goldstein Scale by country
    ax2.bar(country_df['country'], country_df['avg_goldstein_scale'])
    ax2.set_xlabel('Country Code')
    ax2.set_ylabel('Average Goldstein Scale')
    ax2.set_title('Average Goldstein Scale by Country')
    ax2.tick_params(axis='x', rotation=45)
    ax2.axhline(y=0, color='red', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Error analyzing by country: {e}")

## 8. Time Series Analysis

Analyze events over time to identify trends.

In [ ]:
# Time series analysis
time_series_query = """
SELECT 
    SQLDATE as event_date,
    COUNT(*) as daily_events,
    AVG(GoldsteinScale) as avg_goldstein,
    AVG(AvgTone) as avg_tone
FROM gdelt_events
WHERE SQLDATE IS NOT NULL
GROUP BY SQLDATE
ORDER BY SQLDATE DESC
LIMIT 30
"""

try:
    time_series_df = conn.execute(time_series_query).df()
    time_series_df['event_date'] = pd.to_datetime(time_series_df['event_date'], format='%Y%m%d')
    
    print("Recent daily event counts:")
    print(time_series_df.head(10))
    
    # Create time series plot
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10))
    
    # Daily events
    ax1.plot(time_series_df['event_date'], time_series_df['daily_events'], marker='o', linewidth=2)
    ax1.set_ylabel('Daily Events')
    ax1.set_title('GDELT Events Over Time')
    ax1.grid(True, alpha=0.3)
    
    # Average Goldstein Scale
    ax2.plot(time_series_df['event_date'], time_series_df['avg_goldstein'], marker='s', color='orange', linewidth=2)
    ax2.set_ylabel('Average Goldstein Scale')
    ax2.axhline(y=0, color='red', linestyle='--', alpha=0.7)
    ax2.grid(True, alpha=0.3)
    
    # Average Tone
    ax3.plot(time_series_df['event_date'], time_series_df['avg_tone'], marker='^', color='green', linewidth=2)
    ax3.set_ylabel('Average Tone')
    ax3.set_xlabel('Date')
    ax3.axhline(y=0, color='red', linestyle='--', alpha=0.7)
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Error in time series analysis: {e}")

## 9. Interactive Visualization with Plotly

Create interactive visualizations for better data exploration.

In [ ]:
# Create interactive scatter plot
scatter_query = """
SELECT 
    EventCode,
    Actor1CountryCode as country,
    GoldsteinScale,
    AvgTone,
    NumMentions,
    SQLDATE
FROM gdelt_events
WHERE GoldsteinScale IS NOT NULL 
    AND AvgTone IS NOT NULL 
    AND NumMentions IS NOT NULL
    AND Actor1CountryCode IS NOT NULL
LIMIT 1000
"""

try:
    scatter_df = conn.execute(scatter_query).df()
    
    # Create interactive scatter plot
    fig = px.scatter(
        scatter_df, 
        x='GoldsteinScale', 
        y='AvgTone',
        size='NumMentions',
        color='country',
        hover_data=['EventCode', 'SQLDATE'],
        title='GDELT Events: Goldstein Scale vs Average Tone',
        labels={
            'GoldsteinScale': 'Goldstein Scale (Conflict ← → Cooperation)',
            'AvgTone': 'Average Tone (Negative ← → Positive)'
        }
    )
    
    fig.add_hline(y=0, line_dash="dash", line_color="red", opacity=0.5)
    fig.add_vline(x=0, line_dash="dash", line_color="red", opacity=0.5)
    
    fig.show()
    
except Exception as e:
    print(f"Error creating interactive visualization: {e}")

## 10. Custom Query Function

Create a reusable function for custom queries.

In [ ]:
def query_gdelt(query, description="Custom Query"):
    """
    Execute a custom query on the GDELT database.
    
    Args:
        query (str): SQL query to execute
        description (str): Description of the query
    
    Returns:
        pandas.DataFrame: Query results
    """
    try:
        print(f"Executing: {description}")
        result_df = conn.execute(query).df()
        print(f"✅ Query successful. Returned {len(result_df)} rows.")
        return result_df
    except Exception as e:
        print(f"❌ Query failed: {e}")
        return None

# Example usage
custom_query = """
SELECT 
    EventCode,
    COUNT(*) as count,
    AVG(GoldsteinScale) as avg_goldstein
FROM gdelt_events
WHERE GoldsteinScale > 5  -- Focus on cooperative events
GROUP BY EventCode 
ORDER BY count DESC 
LIMIT 10
"""

cooperative_events = query_gdelt(custom_query, "Top cooperative events (Goldstein > 5)")
if cooperative_events is not None:
    print(cooperative_events)

## 11. Data Export Example

Export query results for further analysis.

In [ ]:
# Export sample data for external analysis
export_query = """
SELECT 
    SQLDATE,
    EventCode,
    Actor1CountryCode,
    Actor2CountryCode,
    GoldsteinScale,
    AvgTone,
    NumMentions,
    NumSources
FROM gdelt_events
WHERE SQLDATE >= 20250101  -- Recent data only
LIMIT 1000
"""

try:
    export_df = conn.execute(export_query).df()
    
    # Save to CSV
    output_file = 'gdelt_sample_export.csv'
    export_df.to_csv(output_file, index=False)
    print(f"✅ Data exported to {output_file}")
    print(f"Exported {len(export_df)} records")
    
    # Show basic statistics
    print("\nBasic statistics of exported data:")
    print(export_df.describe())
    
except Exception as e:
    print(f"Error exporting data: {e}")

## 12. Cleanup

Close the database connection.

In [ ]:
# Close the database connection
try:
    conn.close()
    print("✅ Database connection closed successfully")
except Exception as e:
    print(f"Error closing connection: {e}")

## Summary

This notebook demonstrated:

1. **Database Connection**: How to connect to a DuckDB database
2. **Schema Exploration**: Understanding the structure of the gedelt_event table
3. **Data Analysis**: Various analytical queries including:
   - Event type distribution
   - Geographic analysis
   - Time series trends
   - Correlation analysis
4. **Visualization**: Both static (matplotlib) and interactive (plotly) visualizations
5. **Data Export**: Exporting results for further analysis

## Next Steps

- Modify the database path to match your setup
- Adjust column names based on your specific GDELT schema
- Extend the analysis with your specific research questions
- Create custom visualizations for your use case

## GDELT Field Reference

Key GDELT fields used in this analysis:
- **SQLDATE**: Event date in YYYYMMDD format
- **EventCode**: CAMEO event code
- **GoldsteinScale**: Conflict-cooperation scale (-10 to +10)
- **AvgTone**: Average tone of coverage (-100 to +100)
- **Actor1CountryCode/Actor2CountryCode**: ISO country codes
- **NumMentions**: Number of mentions in source material